# Corrected significance tests (camera-ready CR-4)

One family of 12 paired comparisons on the 1,267-row test set, fixed *before* any
p-value was computed (decision D2, 2026-11-01). It tests exactly the claims the paper
makes:

| Group | Comparisons |
|---|---|
| Ablation (LR) | Stage 1 vs 3, Stage 1 vs 4, Stage 3 vs 4 |
| Feature sources (NB) | text+meta vs text, vs metadata-only, vs without job title |
| Model comparison (Table III) | NB text+meta vs LR, SVM, RF, XGBoost (text+meta) |
| DistilBERT (seed 42, pre-declared) | text vs best classical text model; text+meta vs NB text+meta |

"Best classical text model" = the tuned text-only classical configuration with the
highest **CV** macro-F1 (Logistic Regression, no selection; CV 0.605), never the
test-best one.

- **Test:** McNemar's test on paired correct/incorrect predictions, chi-square with
  continuity correction (the same setup as the accepted paper's p-values).
- **Correction:** Holm across all 12 tests.
- **Effect size:** paired bootstrap 95% CI on the macro-F1 difference (A - B), resampling
  test rows with the same 2,000 index draws for every comparison (seed 42).

Every prediction set is first checked against the macro-F1 reported for it, so a
mislabelled column cannot slip through.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

from liar_utils import RANDOM_STATE, load_and_label

test = load_and_label("test.csv")
y = test["Label"].to_numpy()

sources = {
    "ablation": pd.read_csv("ablation_test_predictions_v1.csv"),
    "features": pd.read_csv("metadata_ablation_test_predictions_v1.csv"),
    "models": pd.read_csv("model_comparison_test_predictions_v3_metadata.csv"),
    "distilbert": pd.read_csv("distilbert_v2_test_predictions.csv"),
}
for name, df in sources.items():
    assert df["ID"].tolist() == test["ID"].tolist(), f"{name}: rows not in test.csv order"
    assert (df["y_true"].to_numpy() == y).all(), f"{name}: labels differ from test.csv"

# label -> (source, column, macro-F1 reported elsewhere, for the consistency check)
P = {
    "LR Stage 1 (text)": ("ablation", "pred__1", 0.5965),
    "LR Stage 3 (+selection)": ("ablation", "pred__3", 0.6300),
    "LR Stage 4 (+tuning)": ("ablation", "pred__4", 0.6218),
    "NB text+metadata": ("features", "pred__Naive Bayes__text+metadata", 0.6549),
    "NB text": ("features", "pred__Naive Bayes__text", 0.6035),
    "NB metadata": ("features", "pred__Naive Bayes__metadata", 0.6176),
    "NB text+metadata -job": ("features", "pred__Naive Bayes__text+metadata -job", 0.6316),
    "LR text (tuned)": ("features", "pred__Logistic Regression__text", 0.6153),
    "LR text+metadata": ("models", "pred__Logistic Regression__Text + metadata", None),
    "SVM text+metadata": ("models", "pred__SVM__Text + metadata", None),
    "RF text+metadata": ("models", "pred__Random Forest__Text + metadata", None),
    "XGBoost text+metadata": ("models", "pred__XGBoost__Text + metadata", None),
    "DistilBERT text (seed 42)": ("distilbert", "pred__text__seed42", 0.6435),
    "DistilBERT text+metadata (seed 42)": ("distilbert", "pred__text_meta__seed42", 0.6478),
}
table3 = pd.read_csv("model_comparison_results_v3_metadata.csv").query("Method == 'Text + metadata'").set_index("Model")["Test Macro-F1"]
for label, model in [("LR text+metadata", "Logistic Regression"), ("SVM text+metadata", "SVM"),
                     ("RF text+metadata", "Random Forest"), ("XGBoost text+metadata", "XGBoost")]:
    src, col, _ = P[label]
    P[label] = (src, col, round(table3[model], 4))

pred = {}
for label, (src, col, expected) in P.items():
    pred[label] = sources[src][col].to_numpy()
    got = f1_score(y, pred[label], average="macro")
    assert abs(got - expected) < 5e-4, f"{label}: macro-F1 {got:.4f}, expected {expected}"
nb_models = sources["models"]["pred__Naive Bayes__Text + metadata"].to_numpy()
assert (nb_models == pred["NB text+metadata"]).all(), "NB text+metadata differs between notebooks"
print("All 14 prediction sets verified against their reported macro-F1.")

All 14 prediction sets verified against their reported macro-F1.


In [2]:
FAMILY = [
    ("Ablation", "LR Stage 3 (+selection)", "LR Stage 1 (text)"),
    ("Ablation", "LR Stage 4 (+tuning)", "LR Stage 1 (text)"),
    ("Ablation", "LR Stage 3 (+selection)", "LR Stage 4 (+tuning)"),
    ("Feature sources", "NB text+metadata", "NB text"),
    ("Feature sources", "NB text+metadata", "NB metadata"),
    ("Feature sources", "NB text+metadata", "NB text+metadata -job"),
    ("Models", "NB text+metadata", "LR text+metadata"),
    ("Models", "NB text+metadata", "SVM text+metadata"),
    ("Models", "NB text+metadata", "RF text+metadata"),
    ("Models", "NB text+metadata", "XGBoost text+metadata"),
    ("DistilBERT", "DistilBERT text (seed 42)", "LR text (tuned)"),
    ("DistilBERT", "DistilBERT text+metadata (seed 42)", "NB text+metadata"),
]
assert len(FAMILY) == 12

rng = np.random.RandomState(RANDOM_STATE)
boot_idx = [rng.randint(0, len(y), len(y)) for _ in range(2000)]

rows = []
for group, a, b in FAMILY:
    ca, cb = pred[a] == y, pred[b] == y
    table = [[int(np.sum(ca & cb)), int(np.sum(ca & ~cb))], [int(np.sum(~ca & cb)), int(np.sum(~ca & ~cb))]]
    res = mcnemar(table, exact=False, correction=True)
    diffs = [f1_score(y[i], pred[a][i], average="macro") - f1_score(y[i], pred[b][i], average="macro") for i in boot_idx]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    rows.append({
        "Group": group, "A": a, "B": b,
        "Macro-F1 A": f1_score(y, pred[a], average="macro"), "Macro-F1 B": f1_score(y, pred[b], average="macro"),
        "Diff (A-B)": f1_score(y, pred[a], average="macro") - f1_score(y, pred[b], average="macro"),
        "CI low": lo, "CI high": hi,
        "Only A correct": table[0][1], "Only B correct": table[1][0],
        "McNemar stat": res.statistic, "p raw": res.pvalue,
    })

family = pd.DataFrame(rows)
reject, p_holm, _, _ = multipletests(family["p raw"], alpha=0.05, method="holm")
family["p Holm"] = p_holm
family["Significant (Holm, 0.05)"] = reject
family.to_csv("pvalue_family_camera_ready.csv", index=False)
pd.set_option("display.width", 250)
family.round(4)

,Group,A,B,Macro-F1 A,Macro-F1 B,Diff (A-B),CI low,CI high,Only A correct,Only B correct,McNemar stat,p raw,p Holm,"Significant (Holm, 0.05)"
0,Ablation,LR Stage 3 (+selection),LR Stage 1 (text),0.6300,0.5965,0.0336,0.0064,0.0611,159,122,4.6121,0.0317,0.1588,False
1,Ablation,LR Stage 4 (+tuning),LR Stage 1 (text),0.6218,0.5965,0.0254,-0.0031,0.0548,173,164,0.1899,0.6630,1.0000,False
2,Ablation,LR Stage 3 (+selection),LR Stage 4 (+tuning),0.6300,0.6218,0.0082,-0.0101,0.0269,88,60,4.9257,0.0265,0.1588,False
3,Feature sources,NB text+metadata,NB text,0.6549,0.6035,0.0514,0.0249,0.0777,157,108,8.6943,0.0032,0.0287,True
4,Feature sources,NB text+metadata,NB metadata,0.6549,0.6176,0.0373,0.0141,0.0602,135,86,10.4253,0.0012,0.0124,True
5,Feature sources,NB text+metadata,NB text+metadata -job,0.6549,0.6316,0.0233,0.0080,0.0381,61,34,7.1158,0.0076,0.0535,False
6,Models,NB text+metadata,LR text+metadata,0.6549,0.6229,0.0320,0.0122,0.0523,107,63,10.8765,0.0010,0.0107,True
7,Models,NB text+metadata,SVM text+metadata,0.6549,0.6220,0.0329,0.0131,0.0531,106,61,11.5928,0.0007,0.0079,True
8,Models,NB text+metadata,RF text+metadata,0.6549,0.6256,0.0293,0.0039,0.0538,132,114,1.1748,0.2784,0.8353,False
9,Models,NB text+metadata,XGBoost text+metadata,0.6549,0.6174,0.0375,0.0106,0.0653,158,124,3.8617,0.0494,0.1976,False
